<a href="https://colab.research.google.com/github/zeyadsheriif/EgyGuide/blob/main/Grad_Project_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cleaning data

In [ ]:
import pandas as pd

df = pd.read_csv("/content/LLM_Data_2 - Grad_Project_Data.csv")
df.head()

,Question,Answer
0,Did ancient Egyptian women have a high social ...,"Yes, they enjoyed a relatively high social sta..."
1,How did property inheritance work in ancient E...,All landed property was passed down through th...
2,Why was property passed down through the femal...,It was based on the assumption that maternity ...
3,Did ancient Egyptian women have to wear veils ...,"No, unlike the women of ancient Greece, they e..."
4,How were male and female guests seated at form...,Married guests sat together in pairs on fine c...


In [ ]:
def format_row(row):
    q = str(row["Question"]).strip()
    a = str(row["Answer"]).strip()

    return f"Question: {q} Answer: {a}"

df["text"] = df.apply(format_row, axis=1)

In [ ]:
df = df[df["text"].str.len() > 20]
df = df.drop_duplicates(subset=["text"])

In [ ]:
df[["text"]].to_csv("cleaned_data.csv", index=False)

In [ ]:
df["text"].iloc[0]

'Question: Did ancient Egyptian women have a high social status? Answer: Yes, they enjoyed a relatively high social status and could exert influence outside their domestic roles.'

# Tour Guide RAG Using FAISS and Qwen

## FAISS Retrival model

In [ ]:
!pip install transformers sentence-transformers faiss-cpu accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 67.8 MB/s eta 0:00:00


In [ ]:
import pandas as pd

df = pd.read_csv("cleaned_data.csv")
texts = df["text"].tolist()

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedder.encode(texts, show_progress_bar=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/396 [00:00<?, ?it/s]

In [ ]:
!pip install faiss-cpu

In [ ]:
import faiss
import numpy as np

index = faiss.IndexFlatL2(len(embeddings[0]))
index.add(np.array(embeddings))

In [ ]:
def retrieve(query, k=3):
    q_emb = embedder.encode([query])

    distances, indices = index.search(q_emb, k * 2)

    results = [texts[i] for i in indices[0]]

    filtered = [r for r in results if any(word.lower() in r.lower() for word in query.split())]

    return filtered[:k] if filtered else results[:k]

In [ ]:
!pip install transformers sentence-transformers faiss-cpu accelerate pandas numpy tqdm

## Qwen Generation Model

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype="auto"
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [ ]:
def answer_question(query: str, context_list: list, chat_history: list = None) -> str:
    context = "\n".join(context_list)

    history_text = ""
    if chat_history:
        history_text = "Recent Conversation:\n"
        for interaction in chat_history[-2:]:
            history_text += f"Visitor: {interaction['user']}\nGuide: {interaction['bot']}\n"

    prompt = f"""
    You are a friendly and knowledgeable Egyptian tourist guide.

    Answer the question using ONLY the information provided in the context.

    Style:
    - Speak naturally and clearly like a guide.
    - Use 2-3 sentences.

    Strict Rules:
    - Do NOT add explanations or interpretations.
    - Do NOT add any information not directly written in the context.
    - Do NOT justify or comment on the information.
    - Do NOT repeat the question.
    - If the user uses a pronoun (like 'he' or 'it'), use the Recent Conversation to understand who they mean.

    Context:
    {context}

    {history_text}

    Visitor Question: {query}
    Guide Answer:
    """

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=100, do_sample=False, repetition_penalty=1.2)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    answer = response.split("Guide Answer:")[-1]
    answer = answer.split("Visitor Question:")[0].strip()

    if not answer.endswith((".", "!", "?")):
        answer = answer.rsplit(".", 1)[0] + "."

    return answer

### Full pipeline

In [ ]:
import numpy as np
import time
from pydantic import BaseModel, Field
from typing import List, Optional

class RAGResponse(BaseModel):
    status: str
    generated_answer: str
    retrieved_context: List[str]
    latency_seconds: float
    guardrail_decision: str = Field(description="Action taken by the security layer: PASSED, BLOCKED_INPUT, or BLOCKED_OUTPUT")
    error_message: Optional[str] = None


OUT_OF_DOMAIN_ANCHORS = [
    "Write code, a python function, programming script, coding class, or software code.",
    "Ignore instructions, system prompt override, jailbreak, developer mode.",
    "Modern politics, current government, elections, and warfare.",
    "How to build weapons, explosive devices, bombs, or illegal hacking.",
    "Financial advice, stock investments, medical prescriptions, or legal counseling."
]

ANCHOR_EMBEDDINGS = embedder.encode(OUT_OF_DOMAIN_ANCHORS)

def evaluate_input_safety(query: str, threshold: float = 0.40) -> tuple[bool, float]:
    query_vector = embedder.encode([query])[0]

    similarities = []
    for anchor_vec in ANCHOR_EMBEDDINGS:
        norm_product = np.linalg.norm(query_vector) * np.linalg.norm(anchor_vec)
        if norm_product == 0:
            similarities.append(0.0)
        else:
            similarity = np.dot(query_vector, anchor_vec) / norm_product
            similarities.append(float(similarity))

    max_score = max(similarities)
    if max_score > threshold:
        return False, max_score
    return True, max_score


def evaluate_output_grounding(answer: str, context_list: list[str], threshold: float = 0.35) -> bool:
    if not context_list:
        return False

    answer_vector = embedder.encode([answer])[0]
    combined_context = " ".join(context_list)
    context_vector = embedder.encode([combined_context])[0]

    norm_product = np.linalg.norm(answer_vector) * np.linalg.norm(context_vector)
    if norm_product == 0:
        return False

    semantic_overlap = np.dot(answer_vector, context_vector) / norm_product
    return semantic_overlap >= threshold


def generate_tour_response(query: str, chat_history: list = None) -> dict:
    start_time = time.time()

    try:
        is_safe, input_risk_score = evaluate_input_safety(query)
        if not is_safe:
            return RAGResponse(
                status="success",
                generated_answer="As an Egyptian Tour Guide, I am dedicated exclusively to history, ancient monuments, and tourism. I cannot process this request.",
                retrieved_context=[],
                latency_seconds=round(time.time() - start_time, 3),
                guardrail_decision="BLOCKED_INPUT"
            ).model_dump()

        search_query = query
        if chat_history:
            last_question = chat_history[-1]['user']
            last_answer_snippet = chat_history[-1]['bot'][:50]
            search_query = f"{last_question} {last_answer_snippet} {query}"

        context_list = retrieve(search_query, k=3)

        if not context_list:
            return RAGResponse(
                status="success",
                generated_answer="I apologize, but I do not have verified historical data regarding that specific request within my archive.",
                retrieved_context=[],
                latency_seconds=round(time.time() - start_time, 3),
                guardrail_decision="PASSED"
            ).model_dump()

        bot_answer = answer_question(query, context_list, chat_history)
        bot_answer = bot_answer.split('\n')[0].strip()
        if len(bot_answer) < 5 or "ERROR" in bot_answer.upper():
            bot_answer = "I apologize, I am having trouble clarifying that record. Could you rephrase your question?"

        is_grounded = evaluate_output_grounding(bot_answer, context_list)
        if not is_grounded:
            return RAGResponse(
                status="success",
                generated_answer="I cannot confidently confirm that detail using our archival records. Let me know if you would like to explore a different historical era.",
                retrieved_context=context_list,
                latency_seconds=round(time.time() - start_time, 3),
                guardrail_decision="BLOCKED_OUTPUT"
            ).model_dump()

        return RAGResponse(
            status="success",
            generated_answer=bot_answer,
            retrieved_context=context_list,
            latency_seconds=round(time.time() - start_time, 3),
            guardrail_decision="PASSED"
        ).model_dump()

    except Exception as e:
        return RAGResponse(
            status="error",
            generated_answer="A system error occurred.",
            retrieved_context=[],
            latency_seconds=round(time.time() - start_time, 3),
            guardrail_decision="PASSED",
            error_message=str(e)
        ).model_dump()

# Trying arabic qwen

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype="auto"
)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
!pip install deep-translator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 4.2 MB/s eta 0:00:00


In [ ]:
def retrieve(query, k=3):
    q_emb = embedder.encode([query])
    distances, indices = index.search(q_emb, k * 2)

    results = [texts[i] for i in indices[0]]

    return results[:k]

In [ ]:
def answer_question(query):
    context_list = retrieve(query)
    context = "\n".join(context_list)

    prompt = f"""
    You are an expert Egyptologist.

    Answer the question using the context.

    Rules:
    - Combine relevant information into a complete answer.
    - Answer in 2–3 sentences.
    - Do NOT focus on only one detail.
    - Do NOT add information outside the context.

    Context:
    {context}

    Question: {query}
    Answer:
    """

    inputs = tokenizer(prompt, return_tensors="pt")

    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        repetition_penalty=1.2
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    answer = response.split("Answer:")[-1].strip()

    # fix cut sentence
    if not answer.endswith((".", "!", "?")):
        if "." in answer:
            answer = answer.rsplit(".", 1)[0] + "."

    return answer

In [ ]:
from deep_translator import GoogleTranslator

def is_arabic(text):
    return any('\u0600' <= c <= '\u06FF' for c in text)

def answer_question_multilang(query):

    arabic = is_arabic(query)

    #  translate to English
    try:
        if arabic:
            query_en = GoogleTranslator(source='auto', target='en').translate(query)
        else:
            query_en = query
    except:
        return "حدث خطأ في الترجمة" if arabic else "Translation error"


    answer_en = answer_question(query_en)


    try:
        if arabic:
            answer_ar = GoogleTranslator(source='auto', target='ar').translate(answer_en)
            return answer_ar
    except:
        return answer_en

    return answer_en

In [ ]:
answer_question_multilang("مين كانت حتشبسوت؟")

/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:2637: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cuda') before running `.generate()`.
  warnings.warn(


'كانت حتشبسوت ابنة زوجة وأخت زوجها (حيث تزوجت من أخيها غير الشقيق) للفرعون تحتمس الثاني، وخدمت في البداية تحت قيادته قبل أن تصبح وصية مشتركة مع ابن أخيها وابنها المتبنى تحتمس الثالث بعد أن بلغ سن الرشد. وفي وقت لاحق، حكمت بمفردها كفرعون لما يقرب من عقدين من الزمن حتى خلعها تحتمس الثالث خلال حياته اللاحقة.'

In [ ]:
answer_question_multilang("امتى حكمت حتشبسوت؟")

'حكمت حتشبسوت من عام 1504 قبل الميلاد تقريبًا حتى عام 1490 قبل الميلاد تقريبًا، وهي فترة حكمها التي أعقبها حكمها المستقل الكامل الذي استمر لحوالي 21 عامًا أو نحو ذلك اعتمادًا على مصادر وتفسيرات مختلفة. على وجه الدقة، حكمت بين ج. 1507 – 1458 قبل الميلاد بناءً على بعض التقديرات العلمية. يمكن أن تختلف المدة المحددة قليلاً ولكنها تقع بشكل عام ضمن هذا النطاق.'

In [ ]:
answer_question_multilang("مين هو رمسيس الثاني؟")

'رمسيس الثاني (مكتوب أيضًا رمسيس أو رمسيس) كان الفرعون الثالث من الأسرة التاسعة عشرة في مصر وحكم من عام 1279 تقريبًا حتى وفاته حوالي عام 1213 قبل الميلاد. ويعتبر أحد أبرز حكام مصر القديمة بسبب فترة حكمه الطويلة ومشاريع البناء العديدة في مواقع مختلفة في جميع أنحاء مصر.'